# LeetCode 1061 — Lexicographically Smallest Equivalent String bằng Python

Link đề: https://leetcode.com/problems/lexicographically-smallest-equivalent-string/ · Độ khó: **Medium** · Chủ đề: Union Find (hợp nhất tập rời nhau), String (chuỗi), Graph (đồ thị)

Skeleton LeetCode:

```python
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        ...
```

Ghi chú: hai chuỗi `s1`, `s2` cùng độ dài cho biết `s1[i]` và `s2[i]` là hai **ký tự tương đương**. Quan hệ tương đương có tính phản xạ, đối xứng và **bắc cầu** (nếu `a == b` và `b == c` thì `a == c`). Với chuỗi `baseStr`, ta được phép thay mỗi ký tự bằng bất kỳ ký tự tương đương với nó; hãy trả về chuỗi **nhỏ nhất theo thứ tự từ điển** có thể thu được. Ý tưởng chính: các ký tự tương đương tạo thành các **nhóm** (thành phần liên thông); mỗi ký tự được thay bằng **ký tự nhỏ nhất trong nhóm của nó**. Dùng **Union-Find** với quy tắc: khi hợp nhất hai nhóm, **gốc là ký tự nhỏ hơn**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề và vì sao đáp án là thay **từng ký tự độc lập** bằng ký tự nhỏ nhất trong nhóm tương đương.
2. Nắm các công cụ Python cần thiết: `ord`/`chr`, danh sách `parent` cho Union-Find, `zip`, `"".join(...)`, tập hợp `set`.
3. Hiểu **Union-Find** (hợp nhất tập rời nhau) với **nén đường đi** (path compression) và quy tắc "gốc là phần tử nhỏ nhất của nhóm".
4. Đi qua 4 cách: bao đóng bắc cầu bằng thuật toán Warshall trên ma trận `26 x 26` (brute force, dùng làm oracle), duyệt đồ thị (DFS) tìm các thành phần liên thông, Union-Find với gốc nhỏ nhất (cách nộp chính) và lan truyền nhãn nhỏ nhất đến khi ổn định.
5. Tránh các lỗi kinh điển: chỉ xét tương đương trực tiếp (bỏ qua bắc cầu), hợp nhất theo hướng tùy ý rồi dùng gốc làm đáp án, quên gọi `find` trước khi hợp nhất, `KeyError` với ký tự chỉ có trong `baseStr`, và nhầm "chuỗi nhỏ nhất" với việc sắp xếp lại chuỗi.

## 2. Đề bài & ví dụ

Cho hai chuỗi cùng độ dài `s1`, `s2` và một chuỗi `baseStr`.

Ta nói `s1[i]` và `s2[i]` là **ký tự tương đương**. Ví dụ `s1 = "abc"`, `s2 = "cde"` cho `'a' == 'c'`, `'b' == 'd'`, `'c' == 'e'`.

Các ký tự tương đương tuân theo quy tắc của một quan hệ tương đương:

- **Phản xạ:** `'a' == 'a'`.
- **Đối xứng:** `'a' == 'b'` kéo theo `'b' == 'a'`.
- **Bắc cầu:** `'a' == 'b'` và `'b' == 'c'` kéo theo `'a' == 'c'`.

Ví dụ với thông tin từ `s1 = "abc"`, `s2 = "cde"`, các chuỗi `"acd"` và `"aab"` là chuỗi tương đương của `baseStr = "eed"`, và `"aab"` là chuỗi tương đương nhỏ nhất theo thứ tự từ điển.

Trả về **chuỗi tương đương nhỏ nhất theo thứ tự từ điển** của `baseStr` khi dùng thông tin tương đương từ `s1` và `s2`.

**Ví dụ 1:** `s1 = "parker"`, `s2 = "morris"`, `baseStr = "parser"` -> `"makkek"`. Các nhóm: `[m,p]`, `[a,o]`, `[k,r,s]`, `[e,i]`.

**Ví dụ 2:** `s1 = "hello"`, `s2 = "world"`, `baseStr = "hold"` -> `"hdld"`. Các nhóm: `[h,w]`, `[d,e,o]`, `[l,r]`; chỉ ký tự `o` bị đổi thành `d`.

**Ví dụ 3:** `s1 = "leetcode"`, `s2 = "programs"`, `baseStr = "sourcecode"` -> `"aauaaaaada"`.

**Ràng buộc:**

- `1 <= s1.length, s2.length, baseStr.length <= 1000`
- `s1.length == s2.length`
- `s1`, `s2` và `baseStr` chỉ gồm chữ cái tiếng Anh viết thường.

In [1]:
from typing import List, Dict
from itertools import product
import random, string, time

examples = [
    ("parker", "morris", "parser", "makkek"),
    ("hello", "world", "hold", "hdld"),
    ("leetcode", "programs", "sourcecode", "aauaaaaada"),
]
for s1, s2, base, expected in examples:
    print(f"s1={s1!r} s2={s2!r} baseStr={base!r} -> kỳ vọng {expected!r}")

s1='parker' s2='morris' baseStr='parser' -> kỳ vọng 'makkek'
s1='hello' s2='world' baseStr='hold' -> kỳ vọng 'hdld'
s1='leetcode' s2='programs' baseStr='sourcecode' -> kỳ vọng 'aauaaaaada'


## 3. Kiến thức nền

### 3.1. `ord`, `chr` và chỉ số chữ cái

`ord('a') = 97`, nên `ord(c) - 97` cho chỉ số `0..25`; `chr(i + 97)` đổi ngược lại. Có đúng `26` chữ cái nên ta lưu cấu trúc trong các mảng độ dài `26`.

In [2]:
print(ord("a"), ord("z"), ord("c") - 97, chr(4 + 97))
print([chr(97 + i) for i in range(5)])

97 122 2 e
['a', 'b', 'c', 'd', 'e']


### 3.2. Quan hệ tương đương và nhóm

Từ các cặp `(s1[i], s2[i])` ta có một **đồ thị vô hướng** trên các chữ cái; hai chữ cái tương đương khi và chỉ khi chúng nằm trong **cùng một thành phần liên thông** (nhờ đối xứng và bắc cầu). Mỗi thành phần là một **nhóm**.

In [3]:
s1, s2 = "parker", "morris"
edges = list(zip(s1, s2))
print("các cạnh:", edges)
adj = {}
for a, b in edges:
    adj.setdefault(a, set()).add(b)
    adj.setdefault(b, set()).add(a)
print({k: sorted(v) for k, v in sorted(adj.items())})

các cạnh: [('p', 'm'), ('a', 'o'), ('r', 'r'), ('k', 'r'), ('e', 'i'), ('r', 's')]
{'a': ['o'], 'e': ['i'], 'i': ['e'], 'k': ['r'], 'm': ['p'], 'o': ['a'], 'p': ['m'], 'r': ['k', 'r', 's'], 's': ['r']}


### 3.3. Union-Find (hợp nhất tập rời nhau)

Mỗi phần tử có một `parent`; **gốc** của nhóm là phần tử có `parent[x] == x`. Hai thao tác: `find(x)` đi theo `parent` lên gốc (kèm **nén đường đi** để nhanh hơn), và `union(a, b)` nối gốc của hai nhóm. Nếu **luôn cho gốc là phần tử nhỏ hơn** thì gốc của mỗi nhóm chính là phần tử nhỏ nhất của nhóm.

In [4]:
parent = list(range(6))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]          # nén đường đi
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra < rb:
        parent[rb] = ra                        # gốc là phần tử nhỏ hơn
    elif rb < ra:
        parent[ra] = rb
for a, b in [(4, 2), (5, 4), (3, 1)]:
    union(a, b)
    print(f"union({a}, {b}) -> parent = {parent}")
print("gốc của 5:", find(5), "| gốc của 3:", find(3), "| gốc của 0:", find(0))

union(4, 2) -> parent = [0, 1, 2, 3, 2, 5]
union(5, 4) -> parent = [0, 1, 2, 3, 2, 2]
union(3, 1) -> parent = [0, 1, 2, 1, 2, 2]
gốc của 5: 2 | gốc của 3: 1 | gốc của 0: 0


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `s1 = "parker"`, `s2 = "morris"`.

- Các cặp: `p-m`, `a-o`, `r-r`, `k-r`, `e-i`, `r-s`.
- Nhóm: `{m, p}`, `{a, o}`, `{k, r, s}` (vì `k-r` và `r-s`, bắc cầu cho `k-s`), `{e, i}`.
- Ký tự nhỏ nhất của mỗi nhóm: `m`, `a`, `k`, `e`.
- `baseStr = "parser"`: `p -> m`, `a -> a`, `r -> k`, `s -> k`, `e -> e`, `r -> k` ⇒ `"makkek"` ✓.

Vì sao thay **từng ký tự** bằng ký tự nhỏ nhất là tối ưu? Vì thứ tự từ điển so sánh từng vị trí từ trái sang phải và mỗi vị trí được chọn **độc lập** (mỗi vị trí chỉ phụ thuộc nhóm của ký tự đó), nên chọn ký tự nhỏ nhất ở **mọi** vị trí cho chuỗi nhỏ nhất.

## 5. Cách 1 — Brute Force: bao đóng bắc cầu bằng Warshall

**Ý tưởng.** Đúng theo định nghĩa: quan hệ tương đương là **bao đóng phản xạ, đối xứng, bắc cầu** của các cặp đã cho. Dựng ma trận `eq[i][j]` (`26 x 26`) gồm các cặp đã cho (cả hai chiều) và đường chéo; rồi tính bao đóng bắc cầu bằng thuật toán **Warshall**: với mọi `k, i, j`: `eq[i][j] |= eq[i][k] and eq[k][j]`. Sau đó mỗi chữ cái được thay bằng chữ cái nhỏ nhất `j` có `eq[i][j]`. Không dùng đồ thị hay Union-Find nên là **oracle độc lập**.

In [5]:
def smallest_warshall(s1: str, s2: str, baseStr: str) -> str:
    eq = [[i == j for j in range(26)] for i in range(26)]          # phản xạ
    for a, b in zip(s1, s2):
        i, j = ord(a) - 97, ord(b) - 97
        eq[i][j] = eq[j][i] = True                                  # đối xứng
    for k in range(26):                                             # bắc cầu (Warshall)
        for i in range(26):
            if eq[i][k]:
                for j in range(26):
                    if eq[k][j]:
                        eq[i][j] = True
    rep = [min(j for j in range(26) if eq[i][j]) for i in range(26)]
    return "".join(chr(97 + rep[ord(c) - 97]) for c in baseStr)

for s1, s2, base, expected in examples:
    got = smallest_warshall(s1, s2, base)
    print(f"Warshall = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

Warshall = 'makkek' OK
Warshall = 'hdld' OK
Warshall = 'aauaaaaada' OK


**Độ phức tạp và nút thắt.** `O(26^3 + n)`: chậm vô ích về mặt "hằng số" (`17576` bước cho mỗi test) dù `26` nhỏ; nếu bảng chữ cái lớn (ví dụ Unicode) sẽ không dùng được. Đo thử:

In [6]:
rng = random.Random(1061)
s1 = "".join(rng.choice(string.ascii_lowercase) for _ in range(1000))
s2 = "".join(rng.choice(string.ascii_lowercase) for _ in range(1000))
t0 = time.perf_counter()
for _ in range(100):
    smallest_warshall(s1, s2, s1)
dt = (time.perf_counter() - t0) / 100
print(f"Warshall: {dt * 1000:.2f} ms / lần (bảng chữ cái 26)")

Warshall: 0.97 ms / lần (bảng chữ cái 26)


Nút thắt: ta tính cả những cặp `(i, j)` không cần thiết. Chỉ cần biết **nhóm** của mỗi chữ cái: đây là bài toán **thành phần liên thông** / **Union-Find**, chạy gần `O(n)` bất kể kích thước bảng chữ cái.

## 6. Ý tưởng thuật toán / bước đột phá

**Nhóm = thành phần liên thông.** Coi mỗi chữ cái là một đỉnh, mỗi cặp `(s1[i], s2[i])` là một cạnh. Hai chữ cái tương đương khi và chỉ khi cùng thành phần liên thông. Mỗi chữ cái được thay bằng **chữ cái nhỏ nhất** trong thành phần của nó.

**Union-Find với gốc nhỏ nhất.** Khi hợp nhất hai nhóm có gốc `ra`, `rb`, cho gốc **nhỏ hơn** làm gốc chung: `parent[max(ra, rb)] = min(ra, rb)`. Khi đó `find(c)` luôn trả về phần tử nhỏ nhất trong nhóm của `c`, nên có thể dựng chuỗi kết quả ngay.

| Cách | Ý tưởng |
|---|---|
| 2 | Dựng đồ thị kề; duyệt các chữ cái theo thứ tự tăng dần, DFS gán nhóm cho cả thành phần |
| 3 | Union-Find, gốc luôn là chữ cái nhỏ hơn (nộp) |
| 4 | Lan truyền nhãn nhỏ nhất qua các cạnh cho đến khi không còn thay đổi |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS tìm thành phần liên thông

**Giải thích từng đoạn.**

- Dựng `adj[c]`: tập các chữ cái kề với `c` (cả hai chiều).
- Duyệt chữ cái `c` theo thứ tự tăng dần `a..z`; nếu `c` chưa có nhóm, DFS từ `c` và gán `rep[x] = c` cho mọi `x` trong thành phần. Vì `c` là chữ cái **đầu tiên** của thành phần được duyệt (theo thứ tự tăng dần), `c` chính là chữ cái nhỏ nhất.
- Thay từng ký tự của `baseStr` bằng `rep`.

In [7]:
def smallest_dfs(s1: str, s2: str, baseStr: str) -> str:
    adj = [[] for _ in range(26)]
    for a, b in zip(s1, s2):
        i, j = ord(a) - 97, ord(b) - 97
        adj[i].append(j)
        adj[j].append(i)
    rep = [-1] * 26
    for start in range(26):                         # theo thứ tự tăng dần: start là nhỏ nhất của thành phần của nó
        if rep[start] != -1:
            continue
        rep[start] = start
        stack = [start]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if rep[v] == -1:
                    rep[v] = start
                    stack.append(v)
    return "".join(chr(97 + rep[ord(c) - 97]) for c in baseStr)

for s1, s2, base, expected in examples:
    got = smallest_dfs(s1, s2, base)
    print(f"DFS = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

DFS = 'makkek' OK
DFS = 'hdld' OK
DFS = 'aauaaaaada' OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        adj = [[] for _ in range(26)]                    # đồ thị vô hướng trên 26 chữ cái
        for a, b in zip(s1, s2):
            i, j = ord(a) - 97, ord(b) - 97
            adj[i].append(j)
            adj[j].append(i)
        rep = [-1] * 26                                  # rep[c]: chữ cái nhỏ nhất trong nhóm của c
        for start in range(26):                          # duyệt tăng dần nên start nhỏ nhất trong thành phần của nó
            if rep[start] != -1:
                continue
            rep[start] = start
            stack = [start]
            while stack:
                u = stack.pop()
                for v in adj[u]:
                    if rep[v] == -1:
                        rep[v] = start
                        stack.append(v)
        return "".join(chr(97 + rep[ord(c) - 97]) for c in baseStr)

sol_dfs = lambda a, b, c: Solution().smallestEquivalentString(a, b, c)
for s1, s2, base, expected in examples:
    assert sol_dfs(s1, s2, base) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(n + 26 + E)` với `E = len(s1)`; bộ nhớ `O(26 + E)`.

---

### 7.2. Cách 3 — Union-Find với gốc là chữ cái nhỏ nhất (cách nộp chính)

**Giải thích từng đoạn.**

- `parent = list(range(26))`: ban đầu mỗi chữ cái là nhóm riêng.
- `find(x)`: đi lên theo `parent` đến gốc, kèm nén đường đi (`parent[x] = parent[parent[x]]`).
- Với mỗi cặp `(a, b)`: `ra = find(a)`, `rb = find(b)`; nếu khác nhau, đặt gốc **nhỏ hơn** làm cha của gốc lớn hơn.
- Kết quả: thay mỗi ký tự `c` của `baseStr` bằng `chr(find(c) + 97)`.

In [9]:
def smallest_union_find(s1: str, s2: str, baseStr: str) -> str:
    parent = list(range(26))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in zip(s1, s2):
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        if ra < rb:
            parent[rb] = ra
        elif rb < ra:
            parent[ra] = rb
    return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)

for s1, s2, base, expected in examples:
    got = smallest_union_find(s1, s2, base)
    print(f"Union-Find = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

Union-Find = 'makkek' OK
Union-Find = 'hdld' OK
Union-Find = 'aauaaaaada' OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        parent = list(range(26))                         # parent[x]: cha của chữ cái x (gốc có parent[x] == x)

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]            # nén đường đi
                x = parent[x]
            return x

        for a, b in zip(s1, s2):
            ra, rb = find(ord(a) - 97), find(ord(b) - 97)
            if ra < rb:                                  # gốc luôn là chữ cái nhỏ hơn
                parent[rb] = ra
            elif rb < ra:
                parent[ra] = rb
        return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)

sol = lambda a, b, c: Solution().smallestEquivalentString(a, b, c)
for s1, s2, base, expected in examples:
    assert sol(s1, s2, base) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** mỗi `find` gần `O(alpha(26)) ≈ O(1)`; tổng `O(n)`. Bộ nhớ `O(26)`.

---

### 7.3. Cách 4 — Lan truyền nhãn nhỏ nhất đến khi ổn định

**Ý tưởng khác.** Gán mỗi chữ cái một nhãn ban đầu bằng chính nó. Lặp: với mỗi cặp `(a, b)`, cả hai nhận nhãn `min(nhãn[a], nhãn[b])`. Dừng khi một vòng không đổi gì. Nhãn nhỏ nhất của thành phần lan dần qua các cạnh, và khi ổn định mọi chữ cái trong thành phần có nhãn là chữ cái nhỏ nhất của thành phần. Mỗi vòng `O(E)`, số vòng tối đa `26` (độ dài đường đi lớn nhất).

In [11]:
def smallest_propagation(s1: str, s2: str, baseStr: str) -> str:
    label = list(range(26))
    pairs = [(ord(a) - 97, ord(b) - 97) for a, b in zip(s1, s2)]
    changed = True
    while changed:
        changed = False
        for i, j in pairs:
            m = min(label[i], label[j])
            if label[i] != m:
                label[i] = m
                changed = True
            if label[j] != m:
                label[j] = m
                changed = True
    return "".join(chr(97 + label[ord(c) - 97]) for c in baseStr)

for s1, s2, base, expected in examples:
    got = smallest_propagation(s1, s2, base)
    print(f"lan truyền = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

lan truyền = 'makkek' OK
lan truyền = 'hdld' OK
lan truyền = 'aauaaaaada' OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        label = list(range(26))                          # nhãn ban đầu: chính chữ cái đó
        pairs = [(ord(a) - 97, ord(b) - 97) for a, b in zip(s1, s2)]
        changed = True
        while changed:                                   # lan truyền nhãn nhỏ nhất qua các cạnh cho đến khi ổn định
            changed = False
            for i, j in pairs:
                m = min(label[i], label[j])
                if label[i] != m:
                    label[i] = m
                    changed = True
                if label[j] != m:
                    label[j] = m
                    changed = True
        return "".join(chr(97 + label[ord(c) - 97]) for c in baseStr)

sol_prop = lambda a, b, c: Solution().smallestEquivalentString(a, b, c)
for s1, s2, base, expected in examples:
    assert sol_prop(s1, s2, base) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(26 * E)` trong trường hợp xấu, bộ nhớ `O(26 + E)`.

## 8. TRACE từng bước

### 8.1. Union-Find cho ví dụ 1

In `parent` (chỉ các chữ cái có liên quan) sau mỗi cặp.

In [13]:
def trace_union_find(s1, s2, base):
    parent = list(range(26))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def show():
        return {chr(97 + i): chr(97 + parent[i]) for i in range(26) if parent[i] != i}
    for a, b in zip(s1, s2):
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        if ra == rb:
            print(f"cặp ({a}, {b}): cùng gốc {chr(97 + ra)} -> không đổi; cha khác chính nó: {show()}")
            continue
        small, large = min(ra, rb), max(ra, rb)
        parent[large] = small
        print(f"cặp ({a}, {b}): gốc {chr(97 + ra)} và {chr(97 + rb)} -> gốc {chr(97 + small)} làm cha của {chr(97 + large)}; cha khác chính nó: {show()}")
    result = "".join(chr(find(ord(c) - 97) + 97) for c in base)
    print("thay từng ký tự của", repr(base), "bằng gốc ->", repr(result))
    return result

assert trace_union_find("parker", "morris", "parser") == "makkek"

cặp (p, m): gốc p và m -> gốc m làm cha của p; cha khác chính nó: {'p': 'm'}
cặp (a, o): gốc a và o -> gốc a làm cha của o; cha khác chính nó: {'o': 'a', 'p': 'm'}
cặp (r, r): cùng gốc r -> không đổi; cha khác chính nó: {'o': 'a', 'p': 'm'}
cặp (k, r): gốc k và r -> gốc k làm cha của r; cha khác chính nó: {'o': 'a', 'p': 'm', 'r': 'k'}
cặp (e, i): gốc e và i -> gốc e làm cha của i; cha khác chính nó: {'i': 'e', 'o': 'a', 'p': 'm', 'r': 'k'}
cặp (r, s): gốc k và s -> gốc k làm cha của s; cha khác chính nó: {'i': 'e', 'o': 'a', 'p': 'm', 'r': 'k', 's': 'k'}
thay từng ký tự của 'parser' bằng gốc -> 'makkek'


### 8.2. Chuỗi bắc cầu dài: `a-b`, `b-c`, `c-d`, ...

Nhóm lớn hình thành dần; gốc luôn là `a`.

In [14]:
s1, s2 = "abcde", "bcdef"
print(trace_union_find(s1, s2, "fedcba"))
assert smallest_union_find(s1, s2, "fedcba") == "aaaaaa"

cặp (a, b): gốc a và b -> gốc a làm cha của b; cha khác chính nó: {'b': 'a'}
cặp (b, c): gốc a và c -> gốc a làm cha của c; cha khác chính nó: {'b': 'a', 'c': 'a'}
cặp (c, d): gốc a và d -> gốc a làm cha của d; cha khác chính nó: {'b': 'a', 'c': 'a', 'd': 'a'}
cặp (d, e): gốc a và e -> gốc a làm cha của e; cha khác chính nó: {'b': 'a', 'c': 'a', 'd': 'a', 'e': 'a'}
cặp (e, f): gốc a và f -> gốc a làm cha của f; cha khác chính nó: {'b': 'a', 'c': 'a', 'd': 'a', 'e': 'a', 'f': 'a'}
thay từng ký tự của 'fedcba' bằng gốc -> 'aaaaaa'
aaaaaa


### 8.3. Lan truyền nhãn cho ví dụ 2

In nhãn sau mỗi vòng cho đến khi ổn định.

In [15]:
s1, s2 = "hello", "world"
label = list(range(26))
pairs = [(ord(a) - 97, ord(b) - 97) for a, b in zip(s1, s2)]
involved = sorted({ch for ch in s1 + s2})
print("vòng 0:", {c: chr(97 + label[ord(c) - 97]) for c in involved})
round_no = 0
changed = True
while changed:
    changed = False
    round_no += 1
    for i, j in pairs:
        m = min(label[i], label[j])
        for x in (i, j):
            if label[x] != m:
                label[x] = m
                changed = True
    print(f"vòng {round_no}:", {c: chr(97 + label[ord(c) - 97]) for c in involved}, "(có thay đổi)" if changed else "(ổn định)")
assert "".join(chr(97 + label[ord(c) - 97]) for c in "hold") == "hdld"

vòng 0: {'d': 'd', 'e': 'e', 'h': 'h', 'l': 'l', 'o': 'o', 'r': 'r', 'w': 'w'}
vòng 1: {'d': 'd', 'e': 'e', 'h': 'h', 'l': 'l', 'o': 'd', 'r': 'l', 'w': 'h'} (có thay đổi)
vòng 2: {'d': 'd', 'e': 'd', 'h': 'h', 'l': 'l', 'o': 'd', 'r': 'l', 'w': 'h'} (có thay đổi)
vòng 3: {'d': 'd', 'e': 'd', 'h': 'h', 'l': 'l', 'o': 'd', 'r': 'l', 'w': 'h'} (ổn định)


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (Union-Find với gốc nhỏ nhất): ngắn gọn, `O(n)`, và là kỹ thuật chuẩn cho bài toán "nhóm tương đương".

```python
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        parent = list(range(26))                         # parent[x]: cha của chữ cái x (gốc có parent[x] == x)

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]            # nén đường đi
                x = parent[x]
            return x

        for a, b in zip(s1, s2):
            ra, rb = find(ord(a) - 97), find(ord(b) - 97)
            if ra < rb:                                  # gốc luôn là chữ cái nhỏ hơn
                parent[rb] = ra
            elif rb < ra:
                parent[ra] = rb
        return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)
```

In [16]:
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        parent = list(range(26))                         # parent[x]: cha của chữ cái x (gốc có parent[x] == x)

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]            # nén đường đi
                x = parent[x]
            return x

        for a, b in zip(s1, s2):
            ra, rb = find(ord(a) - 97), find(ord(b) - 97)
            if ra < rb:                                  # gốc luôn là chữ cái nhỏ hơn
                parent[rb] = ra
            elif rb < ra:
                parent[ra] = rb
        return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)

sol = lambda a, b, c: Solution().smallestEquivalentString(a, b, c)
for s1, s2, base, expected in examples:
    got = sol(s1, s2, base)
    print(repr(got), "OK" if got == expected else "SAI")
    assert got == expected

'makkek' OK
'hdld' OK
'aauaaaaada' OK


## 10. Vì sao đúng

**Nhóm tương đương là thành phần liên thông.** Quan hệ tương đương sinh bởi các cặp `(s1[i], s2[i])` là bao đóng phản xạ-đối xứng-bắc cầu của chúng. Hai chữ cái `x`, `y` tương đương khi và chỉ khi tồn tại dãy `x = c_0, c_1, ..., c_t = y` mà mỗi cặp `(c_{k}, c_{k+1})` là một cặp được cho (theo một trong hai chiều): tức `x` và `y` nằm trong cùng thành phần liên thông của đồ thị vô hướng các cặp. (Chiều "nếu" dùng đối xứng và bắc cầu liên tiếp; chiều "chỉ nếu" vì tập các cặp nối được lập thành một quan hệ phản xạ, đối xứng, bắc cầu chứa các cặp đã cho, mà bao đóng là quan hệ nhỏ nhất như vậy.)

**Tối ưu từng vị trí.** Một chuỗi `t` tương đương với `baseStr` khi và chỉ khi `t[i]` tương đương với `baseStr[i]` tại **mọi** vị trí `i` (đề cho phép thay từng ký tự độc lập). Thứ tự từ điển so sánh vị trí đầu tiên khác nhau. Gọi `m_i` là ký tự nhỏ nhất trong nhóm của `baseStr[i]`; chuỗi `m = m_0 m_1 ...` hợp lệ, và với mọi chuỗi hợp lệ `t` ta có `t[i] >= m_i` tại mọi `i`, nên `m <= t` theo thứ tự từ điển. Vậy `m` là đáp án.

**Union-Find.** Bất biến: tại mọi thời điểm, các cây `parent` biểu diễn đúng phân hoạch hiện tại theo các cặp đã xử lý, và **gốc của mỗi cây là phần tử nhỏ nhất của nhóm**. Ban đầu mỗi chữ cái là một nhóm một phần tử, gốc là chính nó (nhỏ nhất). Khi hợp nhất hai nhóm khác nhau có gốc `ra < rb` ta đặt `parent[rb] = ra`: gốc mới `ra` là nhỏ nhất của nhóm thứ nhất, nhỏ hơn mọi phần tử của nhóm thứ hai (vì `rb` là nhỏ nhất của nhóm thứ hai và `ra < rb`), nên là nhỏ nhất của nhóm hợp nhất. Nén đường đi không đổi gốc của cây nên không phá bất biến.

**Cách 2 (DFS).** Duyệt chữ cái theo thứ tự tăng dần; chữ cái `start` đầu tiên chưa có nhóm của một thành phần chính là nhỏ nhất của thành phần đó (mọi chữ cái nhỏ hơn trong cùng thành phần đã được gán nhóm trước, mâu thuẫn). **Cách 4.** Nhãn của mỗi chữ cái luôn là một chữ cái trong cùng thành phần và không bao giờ tăng; khi ổn định, mọi cặp `(i, j)` có nhãn bằng nhau, nên cả thành phần có chung nhãn; nhãn của chữ cái nhỏ nhất `m` không đổi (`m`), nên nhãn chung đó là `m`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) là bao đóng Warshall đúng định nghĩa. Với `baseStr` là cả bảng chữ cái `a..z` thì kết quả chính là **toàn bộ ánh xạ** chữ cái -> đại diện, nên so sánh một chuỗi là đủ để so cả ánh xạ. Ta đối chiếu mọi cách trên **mọi** cặp `(s1, s2)` độ dài `1..3` trên bảng chữ `{a, b, c, d}` và nhiều test ngẫu nhiên.

In [17]:
impls = {"DFS": smallest_dfs, "Union-Find": smallest_union_find, "lan truyền": smallest_propagation,
         "nộp (Cách 3)": sol, "Cách 2 (nộp)": sol_dfs, "Cách 4 (nộp)": sol_prop}
alphabet = string.ascii_lowercase
total = 0
for length in (1, 2, 3):
    for t1 in product("abcd", repeat=length):
        for t2 in product("abcd", repeat=length):
            s1, s2 = "".join(t1), "".join(t2)
            ref = smallest_warshall(s1, s2, alphabet)                 # toàn bộ ánh xạ
            for name, fn in impls.items():
                assert fn(s1, s2, alphabet) == ref, (name, s1, s2)
            total += 1
print("đã đối chiếu", total, "cặp (s1, s2) (độ dài 1..3, bảng chữ {a,b,c,d}) trên toàn bộ ánh xạ: mọi cách khớp oracle")

đã đối chiếu 4368 cặp (s1, s2) (độ dài 1..3, bảng chữ {a,b,c,d}) trên toàn bộ ánh xạ: mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Bao đóng Warshall | `O(26^3 + n)` | `O(26^2)` |
| 2. DFS thành phần liên thông | `O(26 + E + n)` | `O(26 + E)` |
| 3. Union-Find (gốc nhỏ nhất) | `O(E * alpha(26) + n)` | `O(26)` |
| 4. Lan truyền nhãn | `O(26 * E + n)` | `O(26 + E)` |

(`E = len(s1)`, `n = len(baseStr)`, tối đa `1000`.) Đo ở cỡ tối đa với các dạng: ngẫu nhiên, chuỗi bắc cầu dài, toàn một chữ, tất cả đôi một khác nhau:

In [18]:
rng = random.Random(11)
rand = lambda k, letters=string.ascii_lowercase: "".join(rng.choice(letters) for _ in range(k))
cases = {
    "ngẫu nhiên 26 chữ": (rand(1000), rand(1000), rand(1000)),
    "chuỗi bắc cầu a-b, b-c, ...": (string.ascii_lowercase[:-1] * 40, string.ascii_lowercase[1:] * 40, rand(1000)),
    "chuỗi bắc cầu đảo ngược": (string.ascii_lowercase[1:] * 40, string.ascii_lowercase[:-1] * 40, rand(1000)),
    "toàn một chữ": ("a" * 1000, "a" * 1000, rand(1000)),
    "hai chữ": (rand(1000, "mn"), rand(1000, "mn"), rand(1000)),
    "xích dài theo thứ tự xấu (z-y, y-x, ...)": (string.ascii_lowercase[:0:-1] * 40, string.ascii_lowercase[-2::-1] * 40, rand(1000)),
}
for label, (a, b, c) in cases.items():
    ref = sol(a, b, c)
    row = [f"{label:42}"]
    for name, fn in (("Union-Find", sol), ("DFS", sol_dfs), ("lan truyền", sol_prop), ("Warshall", smallest_warshall)):
        t0 = time.perf_counter()
        got = fn(a, b, c)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.3f} ms")
    print(" | ".join(row))

ngẫu nhiên 26 chữ                          | Union-Find   0.541 ms | DFS   0.498 ms | lan truyền   0.530 ms | Warshall   1.265 ms
chuỗi bắc cầu a-b, b-c, ...                | Union-Find   0.508 ms | DFS   0.535 ms | lan truyền   0.530 ms | Warshall   0.776 ms
chuỗi bắc cầu đảo ngược                    | Union-Find   0.507 ms | DFS   0.553 ms | lan truyền   0.511 ms | Warshall   0.813 ms
toàn một chữ                               | Union-Find   0.344 ms | DFS   0.322 ms | lan truyền   0.334 ms | Warshall   0.278 ms
hai chữ                                    | Union-Find   0.421 ms | DFS   0.450 ms | lan truyền   0.448 ms | Warshall   0.314 ms
xích dài theo thứ tự xấu (z-y, y-x, ...)   | Union-Find   0.594 ms | DFS   0.516 ms | lan truyền   0.497 ms | Warshall   0.761 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), test ngẫu nhiên có seed và các tính chất.

In [19]:
def check_all(s1, s2, base, expected=None):
    results = {name: fn(s1, s2, base) for name, fn in impls.items()}
    results["Warshall"] = smallest_warshall(s1, s2, base)
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (name, s1[:20], s2[:20], base[:20], got[:20], first[:20])
    assert len(first) == len(base)
    if expected is not None:
        assert first == expected, (s1, s2, base, first, expected)
    return first

# 12.1 ví dụ đề
for a, b, c, e in examples:
    check_all(a, b, c, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("a", "a", "a", "a"),                          # cặp phản xạ, không đổi
    ("a", "b", "b", "a"),                          # b == a: b đổi thành a
    ("b", "a", "b", "a"),                          # thứ tự cặp không quan trọng (đối xứng)
    ("a", "b", "z", "z"),                          # ký tự không có trong nhóm nào giữ nguyên
    ("abc", "cde", "eed", "aab"),                  # ví dụ trong mô tả đề: nhóm {a, c, e}, {b, d}
    ("ab", "bc", "cba", "aaa"),                    # bắc cầu: a == b == c
    ("bc", "ab", "cba", "aaa"),                    # cùng nhóm, cặp viết theo thứ tự khác
    ("zy", "yx", "xyz", "xxx"),                    # chuỗi bắc cầu ở cuối bảng chữ cái: nhóm {x, y, z}
    ("az", "za", "az", "aa"),                      # hai cặp cùng nhóm {a, z}
    ("aa", "bb", "ab", "aa"),                      # lặp lại cặp
    ("mn", "no", "mno", "mmm"),                    # chuỗi bắc cầu ở giữa
    ("ac", "bd", "abcd", "aacc"),                  # hai nhóm riêng {a, b} và {c, d}
    ("a" * 1000, "a" * 1000, "z" * 1000, "z" * 1000),   # độ dài tối đa, không đổi
    ("ab" * 500, "ba" * 500, "ba" * 500, "aa" * 500),   # độ dài tối đa, một nhóm
]
for a, b, c, e in edge:
    check_all(a, b, c, e)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_one_hop(s1, s2, base):                   # chỉ xét tương đương trực tiếp, bỏ qua bắc cầu
    nbr = {}
    for a, b in zip(s1, s2):
        nbr.setdefault(a, {a}).add(b)
        nbr.setdefault(b, {b}).add(a)
    return "".join(min(nbr.get(c, {c})) for c in base)

def wrong_root_not_min(s1, s2, base):              # hợp nhất theo hướng tùy ý rồi dùng gốc làm đáp án
    parent = list(range(26))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for a, b in zip(s1, s2):
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        if ra != rb:
            parent[ra] = rb                        # luôn gắn a vào b: gốc không nhất thiết nhỏ nhất
    return "".join(chr(find(ord(c) - 97) + 97) for c in base)

def wrong_no_find(s1, s2, base):                   # nối trực tiếp a vào b mà không tìm gốc
    parent = list(range(26))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    for a, b in zip(s1, s2):
        i, j = ord(a) - 97, ord(b) - 97
        if i > j:
            parent[i] = j
        elif j > i:
            parent[j] = i
    return "".join(chr(find(ord(c) - 97) + 97) for c in base)

def wrong_keyerror(s1, s2, base):                  # chỉ tạo ánh xạ cho chữ cái xuất hiện trong s1, s2
    rep = {}
    for a, b in zip(s1, s2):
        for c in (a, b):
            rep.setdefault(c, c)
    for a, b in zip(s1, s2):
        m = min(rep[a], rep[b])
        for c in list(rep):
            if rep[c] in (rep[a], rep[b]):
                rep[c] = m
    return "".join(rep[c] for c in base)

def wrong_sorted(s1, s2, base):                    # nhầm "chuỗi nhỏ nhất" với việc sắp xếp lại chuỗi
    return "".join(sorted(smallest_union_find(s1, s2, base)))

def find_counter(wrong, tries=40000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        L = rng.randint(1, 4)
        s1 = "".join(rng.choice("abcde") for _ in range(L))
        s2 = "".join(rng.choice("abcde") for _ in range(L))
        base = "".join(rng.choice("abcdef") for _ in range(rng.randint(1, 5)))
        if best is not None and (L, len(base)) >= (len(best[0]), len(best[2])):
            continue
        try:
            bad = wrong(s1, s2, base) != smallest_warshall(s1, s2, base)
        except Exception:
            bad = True
        if bad:
            best = (s1, s2, base)
    return best

hand_cases = {                                       # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "chỉ xét tương đương trực tiếp (bỏ bắc cầu)": (wrong_one_hop, "parker", "morris", "parser"),       # ví dụ 1 của đề
    "gốc không phải ký tự nhỏ nhất": (wrong_root_not_min, "a", "b", "ab"),                              # luôn gắn a vào b: gốc là b
    "nối không qua find": (wrong_no_find, "cc", "ab", "abc"),                                            # cặp (c, b) ghi đè liên kết (c, a)
    "KeyError với ký tự không có trong s1, s2": (wrong_keyerror, "a", "b", "bz"),                       # z không xuất hiện trong s1, s2
    "sắp xếp lại chuỗi kết quả": (wrong_sorted, "x", "x", "ba"),                                         # kết quả đúng "ba", sắp xếp thành "ab"
}
counters = {}
for name, (fn, a, b, c) in hand_cases.items():
    try:
        wrong_result = fn(a, b, c)
    except Exception as e:
        wrong_result = e.__class__
    assert wrong_result != smallest_warshall(a, b, c), name
    assert find_counter(fn) is not None, name                    # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = (a, b, c)
    shown = wrong_result.__name__ if isinstance(wrong_result, type) else repr(wrong_result)
    print(f"{name:44} s1={a!r} s2={b!r} baseStr={c!r}: sai -> {shown}, đúng -> {smallest_warshall(a, b, c)!r}")
    check_all(a, b, c)
print("12.3 phản ví dụ: OK")

chỉ xét tương đương trực tiếp (bỏ bắc cầu)   s1='parker' s2='morris' baseStr='parser': sai -> 'makrek', đúng -> 'makkek'
gốc không phải ký tự nhỏ nhất                s1='a' s2='b' baseStr='ab': sai -> 'bb', đúng -> 'aa'


nối không qua find                           s1='cc' s2='ab' baseStr='abc': sai -> 'abb', đúng -> 'aaa'
KeyError với ký tự không có trong s1, s2     s1='a' s2='b' baseStr='bz': sai -> KeyError, đúng -> 'az'


sắp xếp lại chuỗi kết quả                    s1='x' s2='x' baseStr='ba': sai -> 'ab', đúng -> 'ba'
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: nhiều kích thước bảng chữ
rng = random.Random(4061)
for _ in range(4000):
    letters = string.ascii_lowercase[:rng.choice([2, 3, 5, 8, 26])]
    L = rng.randint(1, 12)
    s1 = "".join(rng.choice(letters) for _ in range(L))
    s2 = "".join(rng.choice(letters) for _ in range(L))
    base = "".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(1, 15)))
    check_all(s1, s2, base)
print("12.4 ngẫu nhiên nhỏ: OK")

# 12.5 ngẫu nhiên lớn (độ dài tới 1000)
for _ in range(100):
    letters = string.ascii_lowercase[:rng.choice([2, 4, 10, 26])]
    L = rng.randint(1, 1000)
    s1 = "".join(rng.choice(letters) for _ in range(L))
    s2 = "".join(rng.choice(letters) for _ in range(L))
    base = "".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(1, 1000)))
    check_all(s1, s2, base)
print("12.5 ngẫu nhiên lớn: OK")

# 12.6 tính chất
for _ in range(500):
    L = rng.randint(1, 10)
    s1 = "".join(rng.choice("abcdefgh") for _ in range(L))
    s2 = "".join(rng.choice("abcdefgh") for _ in range(L))
    base = "".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(1, 12)))
    res = sol(s1, s2, base)
    assert all(r <= b for r, b in zip(res, base))                                  # mỗi ký tự không tăng
    assert sol(s1, s2, res) == res                                                 # lặp lại: bất động (idempotent)
    assert sol(s2, s1, base) == res                                                # đổi chỗ s1, s2: không đổi
    assert sol(s1[::-1], s2[::-1], base) == res                                    # đảo thứ tự các cặp: không đổi
    perm = list(zip(s1, s2))
    rng.shuffle(perm)
    assert sol("".join(p[0] for p in perm), "".join(p[1] for p in perm), base) == res   # xáo trộn các cặp: không đổi
    assert sol(s1 + s1, s2 + s2, base) == res                                      # lặp thêm cặp: không đổi
    assert sol(s1, s1, base) == base                                               # cặp tự tương đương: không đổi gì
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: OK
12.5 ngẫu nhiên lớn: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét tương đương trực tiếp, bỏ qua bắc cầu

Chỉ dùng các cặp được cho trực tiếp mà quên rằng `a == b` và `b == c` kéo theo `a == c`.

In [22]:
a, b, c = counters["chỉ xét tương đương trực tiếp (bỏ bắc cầu)"]
print(f"s1={a!r} s2={b!r} baseStr={c!r} | sai ->", repr(wrong_one_hop(a, b, c)), "| đúng ->", repr(sol(a, b, c)))
assert wrong_one_hop(a, b, c) != sol(a, b, c)
print("ví dụ 1 của đề:", wrong_one_hop("parker", "morris", "parser"), "(đúng là makkek)")

s1='parker' s2='morris' baseStr='parser' | sai -> 'makrek' | đúng -> 'makkek'
ví dụ 1 của đề: makrek (đúng là makkek)


Vì sao sai: nhóm `{k, r, s}` trong ví dụ 1 chỉ có `k-r` và `r-s` trực tiếp; `k` và `s` tương đương nhờ bắc cầu. **Đúng:** dùng Union-Find hoặc thành phần liên thông.

### Lỗi 2 — Hợp nhất theo hướng tùy ý rồi dùng gốc làm đáp án

Gốc của nhóm chỉ là đại diện, chưa chắc nhỏ nhất.

In [23]:
a, b, c = counters["gốc không phải ký tự nhỏ nhất"]
print(f"s1={a!r} s2={b!r} baseStr={c!r} | sai ->", repr(wrong_root_not_min(a, b, c)), "| đúng ->", repr(sol(a, b, c)))
assert wrong_root_not_min(a, b, c) != sol(a, b, c)

s1='a' s2='b' baseStr='ab' | sai -> 'bb' | đúng -> 'aa'


Vì sao sai: nếu luôn gắn `a` vào `b` thì gốc là chữ cái xuất hiện ở `s2` cuối cùng chứ không phải nhỏ nhất. **Đúng:** khi hợp nhất, cho **gốc nhỏ hơn** làm cha.

### Lỗi 3 — Nối trực tiếp hai chữ cái mà không tìm gốc

`parent[i] = j` bỏ qua rằng `i` hoặc `j` có thể đã nằm trong nhóm khác; các nhóm bị tách ra.

In [24]:
a, b, c = counters["nối không qua find"]
print(f"s1={a!r} s2={b!r} baseStr={c!r} | sai ->", repr(wrong_no_find(a, b, c)), "| đúng ->", repr(sol(a, b, c)))
assert wrong_no_find(a, b, c) != sol(a, b, c)

s1='cc' s2='ab' baseStr='abc' | sai -> 'abb' | đúng -> 'aaa'


Vì sao sai: nhóm của `i` có gốc `find(i)` khác `i` nên phải nối **gốc** với gốc, nếu không mất liên kết đã có. **Đúng:** `ra, rb = find(i), find(j)` rồi `parent[max(ra, rb)] = min(ra, rb)`.

### Lỗi 4 — `KeyError` với ký tự chỉ có trong `baseStr`

Nếu chỉ lập ánh xạ cho chữ cái xuất hiện trong `s1`, `s2` thì ký tự lạ của `baseStr` làm lỗi.

In [25]:
a, b, c = counters["KeyError với ký tự không có trong s1, s2"]
try:
    wrong_keyerror(a, b, c)
    print("không có lỗi (không mong đợi)")
except Exception as e:
    print(f"s1={a!r} s2={b!r} baseStr={c!r} | sai -> {type(e).__name__}: {e} | đúng ->", repr(sol(a, b, c)))

s1='a' s2='b' baseStr='bz' | sai -> KeyError: 'z' | đúng -> 'az'


Vì sao sai: `baseStr` có thể chứa chữ cái không nằm trong bất kỳ cặp nào; chúng tương đương chỉ với chính mình. **Đúng:** cấu trúc cho đủ `26` chữ cái (mọi chữ cái mặc định là nhóm riêng).

### Lỗi 5 — Nhầm "chuỗi nhỏ nhất" với sắp xếp lại chuỗi

In [26]:
a, b, c = counters["sắp xếp lại chuỗi kết quả"]
print(f"s1={a!r} s2={b!r} baseStr={c!r} | sai ->", repr(wrong_sorted(a, b, c)), "| đúng ->", repr(sol(a, b, c)))
assert wrong_sorted(a, b, c) != sol(a, b, c)

s1='x' s2='x' baseStr='ba' | sai -> 'ab' | đúng -> 'ba'


Vì sao sai: đề chỉ cho phép **thay** ký tự bằng ký tự tương đương, **không** cho đổi vị trí. **Đúng:** thay từng ký tự tại chỗ, giữ nguyên thứ tự.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "các cặp phần tử tương đương, có tính bắc cầu; nhóm các phần tử" ⇒ **Union-Find** (hoặc thành phần liên thông).
- Muốn đại diện nhóm là phần tử nhỏ nhất: khi hợp nhất cho **gốc nhỏ hơn** làm cha.
- Union-Find chuẩn: `find` có nén đường đi, `union` nối **gốc** với gốc.
- Cho tối ưu theo thứ tự từ điển, kiểm tra xem mỗi vị trí có chọn **độc lập** không; nếu có thì chọn nhỏ nhất ở mọi vị trí.
- Bảng chữ cái nhỏ (`26`) cho phép các cách cài đặt đơn giản bằng mảng độ dài `26`.
- Cùng họ: Satisfiability of Equality Equations (990), Number of Provinces (547), Redundant Connection (684), Accounts Merge (721).
- Test nên có: bắc cầu dài, các cặp lặp, chữ cái chỉ có trong `baseStr`, hai nhóm riêng, độ dài `1000`.

## 15. Biến thể

1. **Chuỗi tương đương lớn nhất theo thứ tự từ điển:** gốc là phần tử **lớn nhất** của nhóm.
2. **Đếm số nhóm** tương đương (số thành phần liên thông có ít nhất một chữ cái xuất hiện trong các cặp).
3. **Satisfiability of Equality Equations (990):** bài tập 5.
4. Bài liên quan: 547, 684, 721, 990.

Kiểm biến thể 1 (lớn nhất) bằng oracle Warshall (đổi `min` thành `max`):

In [27]:
def largest_union_find(s1: str, s2: str, baseStr: str) -> str:
    parent = list(range(26))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(s1, s2):
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        if ra < rb:
            parent[ra] = rb                       # gốc là phần tử LỚN hơn
        elif rb < ra:
            parent[rb] = ra
    return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)

def largest_warshall(s1: str, s2: str, baseStr: str) -> str:
    eq = [[i == j for j in range(26)] for i in range(26)]
    for a, b in zip(s1, s2):
        i, j = ord(a) - 97, ord(b) - 97
        eq[i][j] = eq[j][i] = True
    for k in range(26):
        for i in range(26):
            for j in range(26):
                if eq[i][k] and eq[k][j]:
                    eq[i][j] = True
    rep = [max(j for j in range(26) if eq[i][j]) for i in range(26)]
    return "".join(chr(97 + rep[ord(c) - 97]) for c in baseStr)

def count_groups(s1: str, s2: str) -> int:
    parent = list(range(26))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    letters = set()
    for a, b in zip(s1, s2):
        letters.update((a, b))
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return len({find(ord(c) - 97) for c in letters})

rng = random.Random(15)
for _ in range(1000):
    L = rng.randint(1, 8)
    s1 = "".join(rng.choice("abcdefg") for _ in range(L))
    s2 = "".join(rng.choice("abcdefg") for _ in range(L))
    base = "".join(rng.choice("abcdefgh") for _ in range(8))
    assert largest_union_find(s1, s2, base) == largest_warshall(s1, s2, base)
    # số nhóm = số giá trị khác nhau của ánh xạ nhỏ nhất trên các chữ cái xuất hiện
    letters = sorted(set(s1 + s2))
    assert count_groups(s1, s2) == len(set(sol(s1, s2, "".join(letters))))
print("biến thể lớn nhất / đếm nhóm: khớp oracle")

biến thể lớn nhất / đếm nhóm: khớp oracle


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bao đóng Warshall | Theo định nghĩa | `O(26^3)` | `O(26^2)` | Oracle |
| 2. DFS | Thành phần liên thông | `O(26 + E)` | `O(26 + E)` | Dễ hiểu |
| 3. Union-Find | Gốc là chữ cái nhỏ nhất | `O(E)` | `O(26)` | **Cách nộp chính** |
| 4. Lan truyền nhãn | Nhãn nhỏ nhất qua cạnh | `O(26 E)` | `O(26 + E)` | Đơn giản, dễ viết |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(ord("d") - 97, chr(2 + 97))
print(Solution().smallestEquivalentString("ab", "bc", "cab"))
print(Solution().smallestEquivalentString("a", "b", "bz"))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # cặp (số, ký tự)
my_guess_2 = None      # kết quả với "ab", "bc", "cab"
my_guess_3 = None      # kết quả với "a", "b", "bz"

### Bài 2 — Tính tay (dễ-vừa)

`s1 = "dca"`, `s2 = "aeb"`, `baseStr = "ecdb"`. Liệt kê các nhóm và ghi chuỗi kết quả.

In [29]:
# Bài 2: điền danh sách nhóm (mỗi nhóm là chuỗi đã sắp xếp) và kết quả
hand_groups, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành Union-Find với gốc nhỏ nhất.

In [30]:
def exercise_3(s1: str, s2: str, baseStr: str) -> str:
    parent = list(range(26))

    def find(x: int) -> int:
        while parent[x] != x:
            # TODO 1: nén đường đi: parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in zip(s1, s2):
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        # TODO 2: nếu ra < rb thì parent[rb] = ra; nếu rb < ra thì parent[ra] = rb
        pass
    # TODO 3: thay mỗi ký tự c của baseStr bằng chr(find(ord(c) - 97) + 97) và ghép lại
    pass


# for a, b, c, e in examples:
#     assert exercise_3(a, b, c) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 2, tự viết `exercise_4(s1, s2, baseStr)` bằng **DFS tìm thành phần liên thông**.

In [31]:
def exercise_4(s1: str, s2: str, baseStr: str) -> str:
    # TODO
    pass


# for a, b, c, e in examples:
#     assert exercise_4(a, b, c) == e

### Bài 5 — Mini challenge: Satisfiability of Equality Equations (khó)

Cho mảng chuỗi `equations`, mỗi chuỗi có dạng `"a==b"` hoặc `"a!=b"` (`a`, `b` là chữ cái thường). Trả về `True` nếu có thể gán giá trị nguyên cho các biến sao cho **mọi** phương trình đều đúng. Ví dụ:

```python
exercise_5(["a==b", "b!=a"])            -> False
exercise_5(["b==a", "a==b"])            -> True
exercise_5(["a==b", "b==c", "a==c"])    -> True
exercise_5(["a==b", "b!=c", "c==a"])    -> False
exercise_5(["c==c", "b==d", "x!=z"])    -> True
```
Gợi ý: hợp nhất các phương trình `==`, rồi kiểm tra các phương trình `!=` không nối hai chữ cái cùng nhóm.

In [32]:
def exercise_5(equations: List[str]) -> bool:
    # TODO
    pass


def equations_oracle(equations):
    eq = [[i == j for j in range(26)] for i in range(26)]
    for e in equations:
        if e[1:3] == "==":
            i, j = ord(e[0]) - 97, ord(e[3]) - 97
            eq[i][j] = eq[j][i] = True
    for k in range(26):
        for i in range(26):
            for j in range(26):
                if eq[i][k] and eq[k][j]:
                    eq[i][j] = True
    return all(not (e[1:3] == "!=" and eq[ord(e[0]) - 97][ord(e[3]) - 97]) for e in equations)


# assert exercise_5(["a==b", "b!=a"]) is False
# assert exercise_5(["b==a", "a==b"]) is True
# assert exercise_5(["a==b", "b==c", "a==c"]) is True
# assert exercise_5(["a==b", "b!=c", "c==a"]) is False
# assert exercise_5(["c==c", "b==d", "x!=z"]) is True

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `ord('d') = 100`; `chr(99) = 'c'`.
- Mức 2: `"ab","bc"`: nhóm `{a, b, c}` (bắc cầu); với `"a","b"`: nhóm `{a, b}` còn `z` đứng riêng.
- Mức 3: `(3, 'c')`; `"cab"` thành `"aaa"`; `"bz"` thành `"az"`.

**Bài 2**
- Mức 1: các cặp `d-a`, `c-e`, `a-b`.
- Mức 2: nhóm `{a, b, d}` (từ `d-a` và `a-b`) và nhóm `{c, e}`.
- Mức 3: nhỏ nhất của nhóm: `a` và `c`; `"ecdb"` -> `c, c, a, a` ⇒ `"ccaa"`.

**Bài 3**
- Mức 1: `parent[x] = parent[parent[x]]`.
- Mức 2: `if ra < rb: parent[rb] = ra` `elif rb < ra: parent[ra] = rb`.
- Mức 3: `return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)`.

**Bài 4**
- Mức 1: dựng `adj` hai chiều cho `26` chữ cái.
- Mức 2: duyệt `start` từ `0` đến `25`; nếu chưa có nhóm, DFS gán `rep[v] = start`.
- Mức 3: ghép `chr(97 + rep[ord(c) - 97])`.

**Bài 5**
- Mức 1: Union-Find trên `26` chữ cái.
- Mức 2: xử lý mọi phương trình `==` trước (hợp nhất).
- Mức 3: duyệt các phương trình `!=`: nếu `find(a) == find(b)` thì `False`; cuối cùng `True`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
r1 = (ord("d") - 97, chr(2 + 97))
r2 = Solution().smallestEquivalentString("ab", "bc", "cab")
r3 = Solution().smallestEquivalentString("a", "b", "bz")
print(r1, r2, r3)
assert r1 == (3, "c")
assert r2 == "aaa"
assert r3 == "az"

(3, 'c') aaa az


### Đáp án Bài 2

Các cặp `d-a`, `c-e`, `a-b`: nhóm `{a, b, d}` và `{c, e}`. Ký tự nhỏ nhất `a` và `c`. `"ecdb"`: `e -> c`, `c -> c`, `d -> a`, `b -> a` ⇒ `"ccaa"`.

In [34]:
hand_groups_solution, hand_answer_solution = ["abd", "ce"], "ccaa"
assert hand_answer_solution == sol("dca", "aeb", "ecdb") == smallest_warshall("dca", "aeb", "ecdb")
assert sorted(hand_groups_solution) == sorted(
    "".join(sorted(g)) for g in [{c for c in "abcde" if smallest_warshall("dca", "aeb", c) == m} for m in "ac"])
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(s1: str, s2: str, baseStr: str) -> str:
    parent = list(range(26))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in zip(s1, s2):
        ra, rb = find(ord(a) - 97), find(ord(b) - 97)
        if ra < rb:
            parent[rb] = ra
        elif rb < ra:
            parent[ra] = rb
    return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)

for a, b, c, e in examples:
    assert exercise_3_solution(a, b, c) == e
rng = random.Random(3)
for _ in range(500):
    L = rng.randint(1, 8)
    s1 = "".join(rng.choice("abcdefg") for _ in range(L))
    s2 = "".join(rng.choice("abcdefg") for _ in range(L))
    base = "".join(rng.choice(string.ascii_lowercase) for _ in range(10))
    assert exercise_3_solution(s1, s2, base) == smallest_warshall(s1, s2, base)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(s1: str, s2: str, baseStr: str) -> str:
    adj = [[] for _ in range(26)]
    for a, b in zip(s1, s2):
        i, j = ord(a) - 97, ord(b) - 97
        adj[i].append(j)
        adj[j].append(i)
    rep = [-1] * 26
    for start in range(26):
        if rep[start] != -1:
            continue
        rep[start] = start
        stack = [start]
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if rep[v] == -1:
                    rep[v] = start
                    stack.append(v)
    return "".join(chr(97 + rep[ord(c) - 97]) for c in baseStr)

for a, b, c, e in examples:
    assert exercise_4_solution(a, b, c) == e
rng = random.Random(4)
for _ in range(500):
    L = rng.randint(1, 8)
    s1 = "".join(rng.choice("abcdefg") for _ in range(L))
    s2 = "".join(rng.choice("abcdefg") for _ in range(L))
    base = "".join(rng.choice(string.ascii_lowercase) for _ in range(10))
    assert exercise_4_solution(s1, s2, base) == smallest_warshall(s1, s2, base)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(equations: List[str]) -> bool:
    parent = list(range(26))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for e in equations:
        if e[1:3] == "==":
            ra, rb = find(ord(e[0]) - 97), find(ord(e[3]) - 97)
            if ra != rb:
                parent[ra] = rb
    for e in equations:
        if e[1:3] == "!=" and find(ord(e[0]) - 97) == find(ord(e[3]) - 97):
            return False
    return True

def equations_oracle_solution(equations):
    eq = [[i == j for j in range(26)] for i in range(26)]
    for e in equations:
        if e[1:3] == "==":
            i, j = ord(e[0]) - 97, ord(e[3]) - 97
            eq[i][j] = eq[j][i] = True
    for k in range(26):
        for i in range(26):
            for j in range(26):
                if eq[i][k] and eq[k][j]:
                    eq[i][j] = True
    return all(not (e[1:3] == "!=" and eq[ord(e[0]) - 97][ord(e[3]) - 97]) for e in equations)

assert exercise_5_solution(["a==b", "b!=a"]) is False
assert exercise_5_solution(["b==a", "a==b"]) is True
assert exercise_5_solution(["a==b", "b==c", "a==c"]) is True
assert exercise_5_solution(["a==b", "b!=c", "c==a"]) is False
assert exercise_5_solution(["c==c", "b==d", "x!=z"]) is True
rng = random.Random(5)
for _ in range(1000):
    eqs = ["%s%s%s" % (rng.choice("abcdef"), rng.choice(["==", "!="]), rng.choice("abcdef")) for _ in range(rng.randint(1, 8))]
    assert exercise_5_solution(eqs) == equations_oracle_solution(eqs), eqs
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (Union-Find với gốc nhỏ nhất)

```python
class Solution:
    def smallestEquivalentString(self, s1: str, s2: str, baseStr: str) -> str:
        parent = list(range(26))

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]            # nén đường đi
                x = parent[x]
            return x

        for a, b in zip(s1, s2):
            ra, rb = find(ord(a) - 97), find(ord(b) - 97)
            if ra < rb:                                  # gốc luôn là chữ cái nhỏ hơn
                parent[rb] = ra
            elif rb < ra:
                parent[ra] = rb
        return "".join(chr(find(ord(c) - 97) + 97) for c in baseStr)
```

Công thức nhớ nhanh: **nhóm tương đương = thành phần liên thông; mỗi chữ cái → chữ cái nhỏ nhất của nhóm; Union-Find nối gốc với gốc, gốc nhỏ làm cha**.

### Checklist trước khi submit

- [ ] Dùng đủ `26` chữ cái (chữ cái chỉ có trong `baseStr` giữ nguyên)?
- [ ] Nối **gốc** với gốc (`find` trước), không nối trực tiếp hai chữ cái?
- [ ] Gốc luôn là chữ cái **nhỏ hơn** (nên `find(c)` là nhỏ nhất của nhóm)?
- [ ] Thay từng ký tự **tại chỗ**, không sắp xếp lại chuỗi?
- [ ] Đã thử: bắc cầu dài, cặp lặp, hai nhóm riêng, chữ cái lạ trong `baseStr`, độ dài `1000`?